# 面试问题：LLM Serving 怎样正确处理客户端断连、取消、背压与 KV 释放？

**一句话回答。** 把取消设计成请求状态机中的正式事件，而不是直接从队列删一个 ID：接入层用有上下水位的有界队列做背压；调度器区分 queued、running、cancel-requested 与 terminal；abort 必须幂等；已经发射到设备的 step 只能在安全边界收割，期间 KV block 仍归原请求所有；边界到达后丢弃未发布输出、原子释放全部 KV，并且只发出一次 terminal event。所有异步回调都携带 lease/epoch，陈旧回调不能复活请求或重复释放资源。

这里的难点不是调用某个服务框架 API，而是定义清楚所有竞态下的所有权。PagedAttention 把动态 KV 变成可分页管理的 block，但 block 何时安全回收仍由请求生命周期决定。[PagedAttention 论文](https://arxiv.org/abs/2309.06180)说明了 KV 的分页与动态增长；[vLLM issue #26400](https://github.com/vllm-project/vllm/issues/26400)则具体讨论了 model execution 与 `update_from_outputs` 之间处理 pending abort 的时序。下面只用 Python 标准库实现一个可执行的教学状态机，不依赖 vLLM，也不把教学队列参数当作生产默认值。

In [ ]:
from collections import Counter, deque  # 导入计数器与双端队列，用于指标和有界排队。
from enum import Enum  # 导入枚举，使请求状态不依赖容易拼错的裸字符串。
question = "LLM Serving 的取消、背压与 KV 安全回收"  # 保存本实验要回答的核心面试问题。
core_invariants = ("幂等取消", "安全点回收", "唯一终态事件", "陈旧回调隔离", "有界背压")  # 列出实现必须守住的五条不变量。
terminal_states = {"completed", "cancelled", "rejected", "failed"}  # 声明对调用方可见的四类终态。
assert "KV" in question  # 确认题目明确覆盖推理服务中最昂贵的动态资源。
assert len(core_invariants) == 5  # 确认面试回答同时覆盖生命周期、资源和流控。
assert "安全点回收" in core_invariants  # 确认取消不会错误等同于立刻释放设备仍在引用的内存。
assert len(terminal_states) == 4  # 确认终态集合可以用于统一审计。

## 1. 先定义状态机，再讨论 API

请求进入系统后依次经历 `received -> queued -> running`，最后进入完成、取消或失败。客户端断连只表达“调用方不再需要结果”，并不能证明 GPU kernel 已经停止引用 KV，因此 running 请求先进入 `cancel_requested`。queued 请求没有设备工作，可以同步移出队列并结束；running 且没有 in-flight step 的请求也可以立即结束；已经发射 step 的请求则等待 step 边界。

状态迁移表同时是代码审查清单：任何终态都不能回到 running，completed 之后到达的 abort 只能得到幂等结果，queued 到 failed 则覆盖启动时 KV 不足等错误。把非法迁移直接抛错，比在多个布尔字段之间猜测真实状态可靠得多。

In [ ]:
class RequestState(str, Enum):  # 定义可序列化且可比较的请求状态枚举。
    RECEIVED = "received"  # 表示请求已登记但尚未完成准入。
    QUEUED = "queued"  # 表示请求位于有界等待队列中。
    RUNNING = "running"  # 表示请求已经拥有 KV 并可发射推理 step。
    CANCEL_REQUESTED = "cancel_requested"  # 表示取消已接受但资源仍可能被 in-flight step 引用。
    COMPLETED = "completed"  # 表示正常生成达到结束条件。
    CANCELLED = "cancelled"  # 表示取消已经到达安全点并完成资源回收。
    REJECTED = "rejected"  # 表示请求因背压等准入原因未进入执行。
    FAILED = "failed"  # 表示请求因资源或执行错误终止。
VALID_TRANSITIONS = {RequestState.RECEIVED: {RequestState.QUEUED, RequestState.REJECTED, RequestState.CANCELLED}, RequestState.QUEUED: {RequestState.RUNNING, RequestState.CANCEL_REQUESTED, RequestState.FAILED}, RequestState.RUNNING: {RequestState.CANCEL_REQUESTED, RequestState.COMPLETED, RequestState.FAILED}, RequestState.CANCEL_REQUESTED: {RequestState.CANCELLED}}  # 用白名单声明全部合法单向迁移。
TERMINAL = {RequestState.COMPLETED, RequestState.CANCELLED, RequestState.REJECTED, RequestState.FAILED}  # 汇总不会再离开的终态集合。
def transition(record, next_state):  # 对请求执行一次受白名单保护的状态迁移。
    current = record["state"]  # 读取迁移前状态用于验证和审计。
    if next_state not in VALID_TRANSITIONS.get(current, set()):  # 检查目标状态是否出现在当前状态的合法后继中。
        raise ValueError(f"非法迁移: {current.value}->{next_state.value}")  # 对终态复活或跨阶段跳转立即报错。
    record["state"] = next_state  # 只在验证通过后提交新状态。
    record["history"].append(next_state.value)  # 追加状态历史，便于故障重放。
    return next_state  # 返回新状态，方便上层组合流程。
def make_request(request_id, prompt_blocks=1, max_tokens=2):  # 创建包含生命周期与资源字段的最小请求记录。
    return {"id": request_id, "state": RequestState.RECEIVED, "history": [RequestState.RECEIVED.value], "prompt_blocks": prompt_blocks, "max_tokens": max_tokens, "kv_blocks": set(), "generated": [], "cancel_requested": False, "in_flight": False, "lease": 0, "terminal_emitted": False, "terminal_reason": None}  # 返回单一真相源，避免状态散落在多个对象。
probe = make_request("probe")  # 创建一个只用于验证迁移规则的请求。
transition(probe, RequestState.QUEUED)  # 将探针请求推进到排队状态。
transition(probe, RequestState.RUNNING)  # 将探针请求推进到运行状态。
assert probe["history"] == ["received", "queued", "running"]  # 验证迁移历史按真实顺序保留。
illegal_guarded = False  # 初始化非法迁移是否被阻止的标记。
try:  # 尝试从运行状态直接退回排队以构造反例。
    transition(probe, RequestState.QUEUED)  # 执行应被白名单拒绝的逆向迁移。
except ValueError:  # 捕获预期的非法迁移异常。
    illegal_guarded = True  # 记录状态机成功阻止了请求倒退。
assert illegal_guarded  # 验证非法迁移不会静默污染请求状态。
assert probe["state"] is RequestState.RUNNING  # 验证失败迁移没有改写原状态。
assert TERMINAL.isdisjoint({RequestState.RUNNING, RequestState.QUEUED})  # 验证活动状态不会被误判为终态。

## 2. KV block 必须有唯一所有者，release 必须天然幂等

Paged KV 的回收单位是 block，而不是“某请求大概用了多少 token”。资源管理器需要维护 `block -> request_id` 所有权；分配不足时不能部分占用，否则失败路径还要处理半成功事务。释放时按 owner 查出该请求全部 block，并在同一个临界区清除所有权后归还空闲表。

幂等 release 是取消链路的最后一道保险：断连监听器、显式 abort API 和超时任务可能同时提交同一取消；晚到的完成回调也可能重复清理。第一次释放返回真实 block，后续释放返回空集合，绝不能把同一个编号重复放回 free list。教学实现是单线程的，生产实现应在引擎线程串行化这些命令，或使用等价的锁与原子事务。

In [ ]:
class KVBlockPool:  # 定义具有唯一所有权和幂等释放语义的 KV block 池。
    def __init__(self, capacity):  # 初始化固定数量的教学 block。
        self.capacity = capacity  # 保存总容量用于不变量审计。
        self.free = deque(range(capacity))  # 将所有 block 编号放入空闲队列。
        self.owner = {}  # 用 block 到请求 ID 的映射记录唯一所有权。
    def available(self):  # 返回当前可分配的 block 数。
        return len(self.free)  # 空闲队列长度就是立即可用容量。
    def owned_by(self, request_id):  # 查询某请求当前持有的全部 block。
        return tuple(sorted(block for block, owner_id in self.owner.items() if owner_id == request_id))  # 返回排序后的不可变快照便于断言。
    def allocate(self, request_id, count):  # 以全有或全无方式为请求分配 block。
        if count < 0:  # 阻止无意义的负数资源请求。
            raise ValueError("block 数不能为负")  # 对调用方错误立即失败。
        if self.available() < count:  # 在修改所有权前检查完整容量是否足够。
            return ()  # 容量不足时返回空结果且不产生部分分配。
        blocks = tuple(self.free.popleft() for _ in range(count))  # 一次取出请求需要的全部空闲 block。
        for block in blocks:  # 遍历本次成功取得的 block。
            self.owner[block] = request_id  # 将每个 block 的唯一所有者设置为当前请求。
        return blocks  # 返回本次分配结果供请求记录持有。
    def release(self, request_id):  # 幂等释放指定请求拥有的所有 block。
        blocks = self.owned_by(request_id)  # 在修改映射前取得稳定的所有权快照。
        for block in blocks:  # 逐个清除该请求的 block 所有权。
            del self.owner[block]  # 先删除 owner，避免空闲表与所有权表同时声称持有。
            self.free.append(block)  # 再把已解除所有权的 block 放回空闲队列。
        return blocks  # 第一次返回真实集合，重复调用自然返回空集合。
pool_probe = KVBlockPool(4)  # 创建四个 block 的独立探针池。
allocated_probe = pool_probe.allocate("r1", 3)  # 为探针请求一次性分配三个 block。
assert allocated_probe == (0, 1, 2)  # 验证分配结果确定且没有重复编号。
assert pool_probe.available() == 1  # 验证分配后三个 block 不再可用。
assert pool_probe.allocate("r2", 2) == ()  # 验证容量不足时不会只分配一部分。
assert pool_probe.release("r1") == (0, 1, 2)  # 验证首次释放找回该请求全部资源。
assert pool_probe.release("r1") == ()  # 验证重复释放不会再次归还相同 block。
assert pool_probe.available() == pool_probe.capacity  # 验证释放后空闲容量恢复到总容量。

## 3. 背压是准入合同，不是等 OOM 后再报错

无限队列会把过载伪装成越来越长的尾延迟，并让已经断连的请求继续占用内存。接入层应设置硬容量、高水位和低水位：达到高水位后进入 shed 模式，返回可重试的过载结果；只有排空到低水位才恢复，以免在阈值附近反复开关。面向 HTTP 时通常映射为明确的 429/503、`Retry-After` 和 request ID，而不是让连接一直挂起。

队列中的取消要先从队列移除，再进入 cancelled 终态。取消本身也应作为控制命令保证能够进入引擎；如果数据请求塞满的同一队列连 abort 都进不去，系统会在过载时失去主动释放能力。因此生产系统常把控制面命令与普通请求区分优先级。

In [ ]:
class AdmissionQueue:  # 定义带高低水位迟滞的有界准入队列。
    def __init__(self, capacity, high_watermark, low_watermark):  # 初始化硬容量与背压阈值。
        if not 0 <= low_watermark < high_watermark <= capacity:  # 验证低水位、高水位和容量的严格顺序。
            raise ValueError("水位配置非法")  # 拒绝会导致背压无法稳定恢复的配置。
        self.capacity = capacity  # 保存队列硬容量。
        self.high_watermark = high_watermark  # 保存进入 shed 模式的阈值。
        self.low_watermark = low_watermark  # 保存退出 shed 模式的阈值。
        self.items = deque()  # 保存已经通过准入的请求记录。
        self.shedding = False  # 初始状态允许接收新请求。
    def admit(self, record):  # 尝试把 received 请求加入等待队列。
        if self.shedding or len(self.items) >= self.capacity:  # 在高水位迟滞或硬容量满时拒绝新请求。
            transition(record, RequestState.REJECTED)  # 将过载请求推进到可审计的拒绝终态。
            return False  # 告诉调用方本次准入失败。
        transition(record, RequestState.QUEUED)  # 在入队前记录请求已进入排队状态。
        self.items.append(record)  # 把请求追加到 FIFO 队尾。
        if len(self.items) >= self.high_watermark:  # 检查入队后是否达到背压高水位。
            self.shedding = True  # 开启迟滞式流量丢弃保护。
        return True  # 告诉调用方请求已经成功入队。
    def pop(self):  # 取出一个等待调度的请求。
        if not self.items:  # 处理空队列的正常轮询情况。
            return None  # 空队列不产生伪请求。
        record = self.items.popleft()  # 按 FIFO 顺序取出队首请求。
        if len(self.items) <= self.low_watermark:  # 检查排空后是否达到恢复水位。
            self.shedding = False  # 关闭 shed 模式并允许新请求准入。
        return record  # 返回待启动请求但暂不假装已经取得 KV。
    def cancel(self, request_id):  # 从等待队列中移除尚未运行的指定请求。
        original = len(self.items)  # 保存移除前长度用于判断是否命中。
        self.items = deque(record for record in self.items if record["id"] != request_id)  # 重建不含目标请求的 FIFO 队列。
        if len(self.items) <= self.low_watermark:  # 取消也可能让队列降到恢复水位。
            self.shedding = False  # 及时恢复准入而不等待一次调度 pop。
        return len(self.items) != original  # 返回目标请求是否确实曾在队列中。
queue_probe = AdmissionQueue(3, 2, 1)  # 创建容量三、高水位二、低水位一的探针队列。
queue_r1 = make_request("queue-r1")  # 创建第一个准入探针请求。
queue_r2 = make_request("queue-r2")  # 创建第二个准入探针请求。
queue_r3 = make_request("queue-r3")  # 创建第三个准入探针请求。
assert queue_probe.admit(queue_r1)  # 验证低负载下请求正常入队。
assert queue_probe.admit(queue_r2)  # 验证达到高水位的请求本身仍成功入队。
assert queue_probe.shedding  # 验证达到高水位后新流量进入 shed 模式。
assert not queue_probe.admit(queue_r3)  # 验证高水位迟滞期间新请求被明确拒绝。
assert queue_r3["state"] is RequestState.REJECTED  # 验证背压结果成为可观测的请求终态。
assert queue_probe.pop()["id"] == "queue-r1"  # 验证 FIFO 调度并使长度降到低水位。
assert not queue_probe.shedding  # 验证达到低水位后恢复准入。

## 4. 引擎循环把控制命令放在 step 安全边界处理

下面的引擎把记录表、队列、KV block 池、终态事件和指标放在一个单线程所有权域中。`submit` 只负责登记与准入，`start_next` 只有在 KV 全量分配成功后才进入 running，`begin_step` 发放单调递增 lease，`finish_step` 只有持有当前 lease 的回调才可以提交 token 或结束请求。

核心取消规则是：queued 取消立即移除；running 且未发射 step 时立即清理；running 且 in-flight 时只改成 cancel_requested。设备完成后，`finish_step` 先核对 lease，再清除 in-flight；若已取消，则丢弃本 step 的 staged token，释放 KV 并发出唯一终态。这样既不会让用户看到取消后的新 token，也不会在 kernel 仍读写 block 时制造 use-after-free。

In [ ]:
class ServingEngine:  # 定义只依赖标准库的请求生命周期教学引擎。
    def __init__(self, block_count=8, queue_capacity=4, high_watermark=3, low_watermark=1):  # 初始化资源、准入、事件和指标。
        self.pool = KVBlockPool(block_count)  # 创建统一管理 KV 所有权的 block 池。
        self.queue = AdmissionQueue(queue_capacity, high_watermark, low_watermark)  # 创建带迟滞背压的有界队列。
        self.records = {}  # 用请求 ID 保存生命周期单一真相源。
        self.events = []  # 保存只追加一次的终态事件账本。
        self.metrics = Counter()  # 保存低基数计数指标，避免以 request ID 作为标签。
    def _emit_terminal(self, record, reason):  # 为终态请求幂等写入一条完成事件。
        if record["terminal_emitted"]:  # 检查其他竞态路径是否已经发出终态。
            self.metrics["duplicate_terminal_suppressed"] += 1  # 记录被抑制的重复终态尝试。
            return False  # 重复调用不再追加事件。
        if record["state"] not in TERMINAL:  # 确保调用方不会为活动请求伪造完成事件。
            raise ValueError("只有终态可以发出 terminal event")  # 对生命周期实现错误立即失败。
        record["terminal_emitted"] = True  # 在写事件前设置幂等门闩。
        record["terminal_reason"] = reason  # 保存稳定且可聚合的终止原因。
        self.events.append({"sequence": len(self.events) + 1, "request_id": record["id"], "state": record["state"].value, "reason": reason, "generated_tokens": len(record["generated"])})  # 追加携带序号的唯一终态事件。
        self.metrics[f"terminal_{record['state'].value}"] += 1  # 按有限终态名称累计指标。
        return True  # 告诉调用方本次确实发出了事件。
    def submit(self, request_id, prompt_blocks=1, max_tokens=2):  # 登记请求并执行有界准入。
        if request_id in self.records:  # 检查网络重试是否重复使用同一个请求 ID。
            self.metrics["duplicate_submit"] += 1  # 记录幂等提交命中次数。
            return self.records[request_id]  # 返回既有记录而不重复排队。
        record = make_request(request_id, prompt_blocks, max_tokens)  # 创建新的请求生命周期记录。
        self.records[request_id] = record  # 在任何异步控制命令到达前先登记记录。
        self.metrics["submitted"] += 1  # 累计收到的唯一请求数。
        if not self.queue.admit(record):  # 尝试通过有界队列准入请求。
            self.metrics["backpressure_rejected"] += 1  # 累计因背压拒绝的请求数。
            self._emit_terminal(record, "backpressure")  # 为拒绝请求发出唯一终态事件。
        return record  # 返回当前请求记录供调用方查询。
    def start_next(self):  # 启动队首请求并为其原子分配 prompt KV。
        record = self.queue.pop()  # 从 FIFO 队列取得一个待启动请求。
        if record is None:  # 处理调度轮询时没有等待请求的情况。
            return None  # 空轮询不改变任何资源状态。
        blocks = self.pool.allocate(record["id"], record["prompt_blocks"])  # 以全有或全无方式申请 prompt KV block。
        if len(blocks) != record["prompt_blocks"]:  # 检查资源池是否完整满足申请。
            transition(record, RequestState.FAILED)  # 将启动失败请求推进到失败终态。
            self.metrics["kv_admission_failed"] += 1  # 累计启动时 KV 不足次数。
            self._emit_terminal(record, "kv_exhausted")  # 告知调用方请求不会继续运行。
            return None  # 不返回一个没有 KV 的伪运行请求。
        record["kv_blocks"] = set(blocks)  # 在请求记录中保存实际拥有的 block 快照。
        transition(record, RequestState.RUNNING)  # 只有分配成功后才进入 running。
        self.metrics["started"] += 1  # 累计实际进入执行的请求数。
        return record  # 返回可以发射 step 的请求。
    def begin_step(self, request_id):  # 为 running 请求发射一个带防陈旧 lease 的 step。
        record = self.records[request_id]  # 取得目标请求的最新生命周期记录。
        if record["state"] is not RequestState.RUNNING or record["in_flight"]:  # 阻止取消中、终态或已有 step 的请求再次发射。
            return None  # 不合法的发射请求不产生新 lease。
        record["lease"] += 1  # 单调递增执行代次以隔离迟到回调。
        record["in_flight"] = True  # 标记设备可能仍在引用该请求的 KV。
        self.metrics["steps_launched"] += 1  # 累计实际发射的推理 step 数。
        return (record["id"], record["lease"])  # 返回同时绑定请求 ID 和代次的 lease。
    def _finish_cancel(self, record, reason):  # 在确认没有 in-flight step 后完成取消清理。
        if record["state"] is not RequestState.CANCEL_REQUESTED or record["in_flight"]:  # 验证当前确实位于安全释放点。
            return False  # 未到安全点时不释放任何 block。
        released = self.pool.release(record["id"])  # 幂等释放该请求仍拥有的全部 KV block。
        record["kv_blocks"].difference_update(released)  # 同步清除请求侧的所有权快照。
        transition(record, RequestState.CANCELLED)  # 在资源已解除所有权后提交 cancelled 终态。
        self.metrics["cancelled"] += 1  # 累计完成清理的取消请求数。
        self._emit_terminal(record, reason)  # 发出且只发出一次取消终态事件。
        return True  # 告诉调用方取消已在本次调用中完成。
    def abort(self, request_id, source="client_abort"):  # 统一处理显式取消、断连和超时控制命令。
        record = self.records.get(request_id)  # 查询请求是否曾在本引擎登记。
        if record is None:  # 处理取消先于提交或错误 ID 的情况。
            self.metrics["abort_not_found"] += 1  # 累计未知请求取消而不创建幽灵记录。
            return "not_found"  # 返回稳定的幂等控制结果。
        if record["state"] in TERMINAL:  # 检查请求是否已经结束。
            self.metrics["abort_after_terminal"] += 1  # 累计晚到取消用于分析客户端时序。
            return "already_terminal"  # 不修改终态、不重复释放也不重复发事件。
        if record["cancel_requested"]:  # 检查另一个取消来源是否已经赢得竞态。
            self.metrics["duplicate_abort"] += 1  # 累计被合并的重复取消。
            return "already_requested"  # 保持第一次取消的资源处理责任不变。
        record["cancel_requested"] = True  # 设置单调取消门闩，后续路径不能撤销。
        self.metrics["abort_requested"] += 1  # 累计首次接受的取消控制命令。
        if record["state"] is RequestState.QUEUED:  # queued 请求尚未被设备引用。
            self.queue.cancel(request_id)  # 先从等待队列移除，阻止调度器再次取出。
            transition(record, RequestState.CANCEL_REQUESTED)  # 记录取消请求已经进入生命周期。
            self._finish_cancel(record, source)  # 在无 in-flight 的安全点同步完成清理。
            self.metrics["abort_immediate"] += 1  # 累计无需等待 step 的取消。
            return "cancelled"  # 告诉调用方本次已经进入终态。
        if record["state"] is RequestState.RUNNING:  # running 请求可能有设备工作仍在执行。
            transition(record, RequestState.CANCEL_REQUESTED)  # 先阻止后续 step 发射和 token 发布。
            if record["in_flight"]:  # 检查设备是否可能仍持有 KV 引用。
                self.metrics["abort_deferred"] += 1  # 累计必须等待安全点的取消。
                return "deferred_to_step_boundary"  # 明确取消已接受但尚未完成回收。
            self._finish_cancel(record, source)  # 没有 in-flight 时立即完成安全回收。
            self.metrics["abort_immediate"] += 1  # 累计运行态但可立即完成的取消。
            return "cancelled"  # 告诉调用方资源已被回收。
        return "already_requested"  # 为理论上的并发观察窗口提供保守幂等结果。
    def finish_step(self, request_id, lease, staged_token):  # 在设备 step 完成后验证代次并提交或丢弃结果。
        record = self.records.get(request_id)  # 取得请求的最新生命周期记录。
        if record is None or lease != (request_id, record["lease"]) or not record["in_flight"]:  # 拒绝未知请求、旧代次或重复完成回调。
            self.metrics["stale_step_callback"] += 1  # 累计被 fencing 机制隔离的陈旧回调。
            return "stale"  # 陈旧回调不得改写 token、状态或资源所有权。
        record["in_flight"] = False  # 先标记设备已经越过本 step 的安全边界。
        if record["state"] is RequestState.CANCEL_REQUESTED:  # 检查 step 执行期间是否收到取消。
            self.metrics["staged_tokens_discarded"] += 1  # 记录未向客户端发布的 staged token。
            self._finish_cancel(record, "cancelled_at_step_boundary")  # 在安全点释放 KV 并发出取消终态。
            return "cancelled"  # 告诉调度循环不要再安排该请求。
        if record["state"] is not RequestState.RUNNING:  # 防御不符合状态机的完成路径。
            self.metrics["stale_step_callback"] += 1  # 将异常晚到结果归为陈旧回调。
            return "stale"  # 不允许非 running 请求提交生成结果。
        record["generated"].append(staged_token)  # 只有当前 lease 且未取消时才发布本 step token。
        self.metrics["tokens_committed"] += 1  # 累计真正进入响应流的 token 数。
        if len(record["generated"]) >= record["max_tokens"]:  # 检查是否达到本教学请求的结束条件。
            transition(record, RequestState.COMPLETED)  # 将请求推进到正常完成终态。
            released = self.pool.release(request_id)  # 在没有 in-flight 后幂等释放全部 KV。
            record["kv_blocks"].difference_update(released)  # 同步请求侧资源快照。
            self._emit_terminal(record, "max_tokens")  # 为正常完成发出唯一终态事件。
            return "completed"  # 告诉调度循环请求已经结束。
        return "committed"  # 告诉调度循环可以继续安排下一个 step。

## 5. 场景一：排队期间断连，应立即取消且不碰 KV

客户端可能在请求仍排队时关闭 SSE/WebSocket/HTTP 连接。网关应把断连转换成带 request ID 的 abort 控制命令；引擎移除队列项、写入 cancelled 并发终态。因为该请求从未 start，所以它没有 KV，回收动作应为空操作。

重复 abort 可能来自显式取消按钮与断连监听器。调用方需要稳定结果，但引擎不能为第二次请求再写一条 terminal event。这里让第二次返回 `already_terminal`；生产 API 也可以统一返回 200/204，只要内部语义和可观测性明确。

In [ ]:
engine = ServingEngine(block_count=6, queue_capacity=4, high_watermark=3, low_watermark=1)  # 创建用于取消竞态演示的教学引擎。
queued_a = engine.submit("request-a", prompt_blocks=2, max_tokens=2)  # 提交稍后将运行的第一个请求。
queued_b = engine.submit("request-b", prompt_blocks=2, max_tokens=2)  # 提交将在队列中断连的第二个请求。
free_before_queued_abort = engine.pool.available()  # 记录排队请求取消前的空闲 KV 数。
queued_abort_result = engine.abort("request-b", source="client_disconnect")  # 将客户端断连映射为统一 abort 命令。
duplicate_queued_abort = engine.abort("request-b", source="explicit_abort")  # 模拟显式取消与断连监听器重复到达。
assert queued_abort_result == "cancelled"  # 验证 queued 请求不需要等待设备安全点。
assert queued_b["state"] is RequestState.CANCELLED  # 验证排队请求最终进入 cancelled 终态。
assert all(record["id"] != "request-b" for record in engine.queue.items)  # 验证取消请求已从 FIFO 队列移除。
assert engine.pool.available() == free_before_queued_abort  # 验证从未运行的请求不会错误改变 KV 容量。
assert duplicate_queued_abort == "already_terminal"  # 验证重复 abort 返回稳定幂等结果。
assert sum(event["request_id"] == "request-b" for event in engine.events) == 1  # 验证多个取消来源只产生一条终态事件。
assert engine.records["request-b"]["terminal_reason"] == "client_disconnect"  # 验证第一次赢得竞态的取消来源被保留。

## 6. 场景二：step 已经发射，取消只能延迟到边界

当 forward/decode step 已提交到 GPU，CPU 上把状态改成 cancelled 并不等于设备立刻停止。此时提前把 block 放回 free list，另一个请求可能获得同一编号并写入，原 kernel 随后继续访问它，就形成资源层面的 use-after-free。正确策略是把状态推进到 cancel_requested、禁止新 step，并保留所有权。

step 回来后先完成同步，再检查取消门闩。本 step 产出的 token 只是 staged output，不进入客户端流；随后一次性释放 KV，写取消终态。vLLM issue #26400 对“model execution 完成后、处理 outputs 前插入 pending abort”的讨论，本质上就是这个状态观察顺序。

In [ ]:
running_a = engine.start_next()  # 从队列启动 request-a 并分配其 prompt KV。
lease_a = engine.begin_step("request-a")  # 发射一个设备可能正在执行的推理 step。
owned_during_step = engine.pool.owned_by("request-a")  # 记录 in-flight 期间 request-a 的 KV 所有权。
free_during_step = engine.pool.available()  # 记录 in-flight 期间剩余空闲容量。
deferred_result = engine.abort("request-a", source="client_disconnect")  # 在 step 尚未回收时提交取消。
assert running_a["state"] is RequestState.CANCEL_REQUESTED  # 验证取消先进入中间态而非伪装成已清理。
assert deferred_result == "deferred_to_step_boundary"  # 验证 API 明确区分接受取消与完成回收。
assert engine.pool.owned_by("request-a") == owned_during_step  # 验证 in-flight 期间 KV 所有权没有提前解除。
assert engine.pool.available() == free_during_step  # 验证取消事件本身没有把仍被设备引用的 block 放回池中。
boundary_result = engine.finish_step("request-a", lease_a, "不应发布的词元")  # 收割 step 并在安全边界观察取消门闩。
assert boundary_result == "cancelled"  # 验证 step 边界完成了延迟取消。
assert running_a["generated"] == []  # 验证取消后返回的 staged token 没有进入响应流。
assert running_a["state"] is RequestState.CANCELLED  # 验证资源清理后才提交取消终态。
assert engine.pool.owned_by("request-a") == ()  # 验证 request-a 的全部 KV 已在安全点释放。
assert engine.pool.available() == engine.pool.capacity  # 验证两个已取消请求没有遗留 KV 泄漏。
assert sum(event["request_id"] == "request-a" for event in engine.events) == 1  # 验证安全点取消只发出一条终态事件。

## 7. lease/epoch 解决完成、取消和旧回调的 ABA 竞态

仅比较 request ID 不够。异步系统里，一个 step 的完成消息可能重试、乱序甚至在请求已结束后到达；若 request ID 被业务重用，问题更严重。每次发射都递增 lease，完成回调必须同时匹配请求 ID、当前 lease 和 `in_flight=True`。旧 lease 只能增加陈旧回调指标，不能提交 token，也不能清除新 step 的 in-flight 标志。

下面先正常提交第一个 token，再发射第二个 step；当第一步的重复回调晚到时，它被 fencing。第二步的当前回调正常结束请求、释放资源、写 terminal event。完成后的 abort 和重复完成同样只能得到幂等结果。

In [ ]:
request_c = engine.submit("request-c", prompt_blocks=1, max_tokens=2)  # 提交一个用于验证 lease 防陈旧机制的请求。
engine.start_next()  # 启动 request-c 并取得一个 KV block。
lease_c1 = engine.begin_step("request-c")  # 发射 request-c 的第一步并取得第一代 lease。
assert engine.finish_step("request-c", lease_c1, "甲") == "committed"  # 验证当前 lease 可以提交第一个 token。
lease_c2 = engine.begin_step("request-c")  # 发射第二步并递增执行代次。
generated_before_stale = tuple(request_c["generated"])  # 保存旧回调到达前已经发布的 token 快照。
assert engine.finish_step("request-c", lease_c1, "迟到") == "stale"  # 验证第一步重复回调被 lease fencing 拒绝。
assert tuple(request_c["generated"]) == generated_before_stale  # 验证陈旧回调没有污染输出流。
assert request_c["in_flight"]  # 验证陈旧回调没有错误清除第二步的设备占用标记。
assert engine.finish_step("request-c", lease_c2, "乙") == "completed"  # 验证当前第二代 lease 正常完成请求。
assert request_c["generated"] == ["甲", "乙"]  # 验证只有两个当前代次结果被发布。
assert request_c["state"] is RequestState.COMPLETED  # 验证请求进入不可复活的正常完成终态。
assert engine.abort("request-c", source="late_disconnect") == "already_terminal"  # 验证完成后晚到断连不能改写结果。
assert engine.finish_step("request-c", lease_c2, "重复") == "stale"  # 验证重复完成回调不能二次提交或二次释放。
assert engine.pool.owned_by("request-c") == ()  # 验证正常完成路径同样释放全部 KV。
assert sum(event["request_id"] == "request-c" for event in engine.events) == 1  # 验证完成、断连和重复回调竞争时仍只有一个终态事件。

## 8. 指标围绕生命周期时间与资源不变量设计

至少监控：队列深度与高水位驻留时间、准入拒绝率、首次取消请求数、立即/延迟取消数、从 abort 到安全回收的 step 数或时延、被丢弃 staged token、陈旧回调、终态分布、活动请求 KV block、释放 block 数与 KV 泄漏。请求 ID 进入日志和 trace，不应成为 Prometheus 标签，否则会造成高基数灾难。

告警不能只看平均取消延迟。若 p99 abort-to-terminal 上升，同时 in-flight 数和 step duration 上升，可能是大 batch 或慢 kernel；若 cancelled 已增加但 free KV 不回升，优先怀疑 ownership 清理；若 abort-after-terminal 激增，可能是完成事件到网关传播慢。最后应定期扫描三个不变量：每个 block 只有一个 owner；每个 terminal 请求没有 block；每个请求最多一条 terminal event。

In [ ]:
def audit_engine(engine_value):  # 汇总引擎生命周期、事件与 KV 所有权不变量。
    terminal_records = [record for record in engine_value.records.values() if record["state"] in TERMINAL]  # 找出全部已经结束的请求。
    event_counts = Counter(event["request_id"] for event in engine_value.events)  # 统计每个请求对应的终态事件数。
    terminal_with_blocks = [record["id"] for record in terminal_records if engine_value.pool.owned_by(record["id"])]  # 找出终态后仍持有 KV 的泄漏请求。
    duplicate_terminal_ids = [request_id for request_id, count in event_counts.items() if count != 1]  # 找出终态事件不是恰好一次的请求。
    return {"queue_depth": len(engine_value.queue.items), "kv_free": engine_value.pool.available(), "terminal_count": len(terminal_records), "terminal_with_blocks": terminal_with_blocks, "duplicate_terminal_ids": duplicate_terminal_ids, "states": Counter(record["state"].value for record in engine_value.records.values())}  # 返回只含低基数聚合量的审计报告。
report = audit_engine(engine)  # 对取消、完成与陈旧回调场景执行统一审计。
pressure_engine = ServingEngine(block_count=2, queue_capacity=2, high_watermark=1, low_watermark=0)  # 创建一入队就触发背压的独立引擎。
pressure_first = pressure_engine.submit("pressure-first", prompt_blocks=1, max_tokens=1)  # 提交第一个可准入请求并触发高水位。
pressure_second = pressure_engine.submit("pressure-second", prompt_blocks=1, max_tokens=1)  # 提交第二个请求以验证明确拒绝。
pressure_engine.start_next()  # 启动第一个请求并让队列降到低水位。
immediate_running_abort = pressure_engine.abort("pressure-first", source="deadline")  # 在没有 in-flight step 时取消 running 请求。
pressure_report = audit_engine(pressure_engine)  # 审计背压和立即取消场景的资源状态。
assert report["queue_depth"] == 0  # 验证主场景没有遗留幽灵队列项。
assert report["terminal_with_blocks"] == []  # 验证所有终态请求均已解除 KV 所有权。
assert report["duplicate_terminal_ids"] == []  # 验证每个终态请求恰好对应一条事件。
assert report["states"] == Counter({"cancelled": 2, "completed": 1})  # 验证主场景终态分布符合两次取消和一次完成。
assert engine.metrics["abort_deferred"] == 1  # 验证设备执行期间的取消被单独计数。
assert engine.metrics["staged_tokens_discarded"] == 1  # 验证取消边界丢弃的 token 可被观测。
assert engine.metrics["stale_step_callback"] == 2  # 验证两个重复或旧代次回调均被 fencing。
assert pressure_second["state"] is RequestState.REJECTED  # 验证过载请求得到明确可重试的拒绝终态。
assert immediate_running_abort == "cancelled"  # 验证无 in-flight 的 running 请求可以立即安全回收。
assert pressure_first["state"] is RequestState.CANCELLED  # 验证 deadline 取消进入正式终态。
assert pressure_report["kv_free"] == pressure_engine.pool.capacity  # 验证立即取消后 KV 容量完全恢复。
assert pressure_engine.metrics["backpressure_rejected"] == 1  # 验证准入拒绝率具有独立指标。
assert pressure_engine.metrics["abort_immediate"] == 1  # 验证无需等待 step 的取消具有独立指标。
assert all(event["state"] in terminal_states for event in engine.events + pressure_engine.events)  # 验证事件账本只包含预先声明的终态。

## 9. 面试收束：按“准入—取消—安全点—回收—通知”回答

一套完整回答可以这样组织：第一，有界队列在高水位拒绝新流量、低水位恢复，返回明确 retry 合同，控制命令不能被普通流量饿死。第二，用 request ID 和单调状态机统一显式 abort、客户端断连、deadline 与内部故障；abort 是幂等门闩。第三，queued 请求立即移除，running 请求若没有 in-flight step 立即清理，有 in-flight 时只标记 cancel_requested，绝不提前复用 KV。

第四，每个设备 step 带 lease/epoch；只在当前 step 边界提交输出。取消已到达时丢弃 staged token，幂等释放该 request 拥有的全部 block，再发唯一 terminal event。陈旧完成、重复 abort 和完成后断连都不能复活请求、重复返回 block 或重复通知。第五，用 abort-to-terminal 延迟、背压拒绝率、deferred abort、stale callback、KV ownership 与终态事件唯一性闭环监控，并通过故障注入覆盖“abort 恰好发生在 forward 与 output update 之间”的竞态。

生产实现还要补充锁或单线程命令队列、分布式 worker acknowledgement、连接层写失败、流式响应 close、模型并行 rank 同步、进程崩溃后的租约回收和 admission fairness。本 Notebook 的价值是把这些扩展建立在可验证的不变量上：**取消可以立即被接受，但只有资源使用者确认越过安全边界后，KV 才能被重新分配。**